In [1]:
#!/usr/bin/env python3
import os
import sys
import glob
import re
import numpy as np
import pandas as pd
import h5py
import matplotlib
import matplotlib.pyplot as plt
import logging

import torch

# SCUNet imports
module_dir_scu = "/global/u2/k/kberard/SCGSR/Research/Diamond/stock_models/SCUNet"
sys.path.insert(0, module_dir_scu)
from models.network_scunet import SCUNet as SCUNet

# FFT imports
qmc_algo_path = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/qmc_algo_tools')
dev = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/developer_tools')
sys.path.insert(0, qmc_algo_path)
sys.path.insert(0, dev)
from qmc_algo_tools.density_denoise import DensityFourierFilterErrorCeil

# ==========================================
# 0. CONFIGURATION & PATCHES
# ==========================================
model_samp = str(240700000)
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/dmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5'

# CPU-loading patch
_original_torch_load = torch.load
def torch_load_cpu(*args, **kwargs):
    if 'map_location' not in kwargs:
        kwargs['map_location'] = torch.device('cpu')
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = torch_load_cpu
torch.serialization.add_safe_globals([SCUNet])

# Set Device Globally
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==========================================
# 1. MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def transform(density, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return (density - density_ref) / np.sqrt(np.abs(density_ref))
    raise RuntimeError('Unsupported transform type')

def inverse_transform(density_trans, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return density_ref + np.sqrt(np.abs(density_ref)) * density_trans
    raise RuntimeError('Unsupported transform type')

def encode_voxel_to_rgb_global(vol_3d):
    """Normalizes the ENTIRE 3D volume, preventing slice-by-slice distortion."""
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    
    normed = (vol_3d - v_min) / (v_max - v_min)
    rgb_volume = np.stack([normed]*3, axis=-1).astype(np.float32)
    return rgb_volume, v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    """Restores the 3D volume using the global scalars."""
    gray = rgb_volume[:, :, :, 0]
    return gray * (v_max - v_min) + v_min

# ==========================================
# 2. INFERENCE DISPATCHER
# ==========================================
def denoise_with_scunet(rgb_image_np, model):
    img = np.clip(rgb_image_np.astype(np.float32), 0, 1)
    
    # --- ADDED: Dynamic padding to handle arbitrary grid dimensions ---
    h, w, c = img.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    if pad_h > 0 or pad_w > 0:
        img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode='edge')
    # ------------------------------------------------------------------

    img_tensor = torch.from_numpy(np.transpose(img, (2, 0, 1))).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output_tensor = model(img_tensor)
    output_np = output_tensor.squeeze().cpu().detach().numpy()
    if output_np.ndim == 3:
        output_np = np.transpose(output_np, (1, 2, 0))
        
    # --- ADDED: Crop back to original dimensions (84, 112) ---
    if pad_h > 0 or pad_w > 0:
        output_np = output_np[:h, :w, :]
    # ------------------------------------------------------------------
        
    return np.clip(output_np, 0, 1)

def run_model_inference(test_d, ref_d_dft, model_name, models_dict):
    """Routes the input data to the appropriate model logic."""
    if model_name == 'scunet_pre_raw':
        # Pipeline: Raw -> Encode -> SCUNet -> Decode
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(test_d)
        denoised_rgb = np.zeros_like(rgb_vol)
        
        for i in range(test_d.shape[0]):
            denoised_rgb[i] = denoise_with_scunet(rgb_vol[i], models_dict['scunet_pre'])
            
        return decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'scunet_pre_trans':
        # Pipeline: Transform -> Encode -> SCUNet -> Decode -> Inverse Transform
        transformed_d = transform(test_d, ref_d_dft, 'residual_noise')
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(transformed_d)
        denoised_rgb = np.zeros_like(rgb_vol)
        
        for i in range(transformed_d.shape[0]):
            denoised_rgb[i] = denoise_with_scunet(rgb_vol[i], models_dict['scunet_pre'])
            
        decoded_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)
        return inverse_transform(decoded_trans, ref_d_dft, 'residual_noise')

    elif model_name == 'FFT':
        dm = DensityFourierFilterErrorCeil(density_ref=ref_d_dft, filter_mode='augment')
        return dm.denoise(test_d)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

def evaluate_and_enforce(denoised_d, ref_d):
    """Enforces non-negativity and strictly normalizes to 50 electrons before scoring."""
    denoised_d = np.maximum(denoised_d, 0.0)
    # Change from 20.0 to 50.0
    denoised_d = denoised_d * (50.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 3. UNIFIED LOG-LOG PLOTTING
# ==========================================
def generate_log_plots(results_dict, results_ref, DFT_vs_dmc, models_to_run):
    print("\n=== Generating Master Plots & Dataframes ===")
    
    # 1. Compile Dataframe
    df = pd.DataFrame(np.array(sorted(results_ref, key=lambda x: x[0])), columns=["Samples", "Noisy_dmc"])
    for model_name in models_to_run:
        model_data = np.array(sorted(results_dict[model_name], key=lambda x: x[0]))
        temp_df = pd.DataFrame(model_data, columns=["Samples", model_name])
        df = pd.merge(df, temp_df, on="Samples", how="outer")

    df = df.sort_values("Samples").reset_index(drop=True)
    df.to_csv("image_models_performance.csv", index=False)
    print("Data saved to image_models_performance.csv")

    # 2. Plotting Formatting
    matplotlib.use('Agg')
    logging.getLogger('matplotlib').setLevel(logging.WARNING)
    plt.rcParams.update({
        'font.size': 14, 'font.family': 'serif', 'axes.labelsize': 16,
        'axes.linewidth': 1.5, 'xtick.major.size': 7, 'xtick.major.width': 1.5,
        'ytick.major.size': 7, 'ytick.major.width': 1.5,
        'legend.frameon': True, 'legend.edgecolor': 'black', 'legend.fontsize': 10
    })

    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot Baselines
    ax.plot(df["Samples"], df["Noisy_dmc"], marker="o", linestyle="--", color="red", label="Noisy dmc", alpha=0.7, markersize=8)
    ax.axhline(DFT_vs_dmc, color="black", linestyle=":", label="DFT Baseline", linewidth=2)

    # Plot Models
    ax.plot(df["Samples"], df["FFT"], marker="D", linestyle="-", color="magenta", label="FFT", linewidth=2.5, markersize=8)
    ax.plot(df["Samples"], df["scunet_pre_raw"], marker="s", linestyle="-", color="blue", label="SCUNet (Raw)", linewidth=2.5, markersize=8)
    ax.plot(df["Samples"], df["scunet_pre_trans"], marker="^", linestyle="-", color="green", label="SCUNet (Transformed)", linewidth=2.5, markersize=8)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Number of dmc Samples', fontweight='bold', labelpad=10)
    ax.set_ylabel(r'$D_{JS}$ (Jensen-Shannon Divergence)', fontweight='bold', labelpad=10)
    ax.legend(loc='best')
    ax.grid(False)

    plt.tight_layout()
    output_base = "image_models_denoising_convergence"
    plt.savefig(f"{output_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_base}.pdf", bbox_inches='tight')
    print(f"Publication-ready plots saved as {output_base}.png and {output_base}.pdf")

# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (50.0 / np.sum(ref_d_mean))  # Change 20.0 -> 50.0
    
    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (50.0 / np.sum(dft_d))                  # Change 20.0 -> 50.0
        
    DFT_vs_dmc = D_JS(ref_d_mean, dft_d)
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_dmc_mix_mean*.h5")))
    print(f"Found {len(noisy_files)} noisy files.")

    models_to_run = ['scunet_pre_raw', 'scunet_pre_trans', 'FFT']
    
    print(f"Pre-loading SCUNet into Memory on {DEVICE}...")
    models_dict = {}
    m = SCUNet(in_nc=3, config=[4, 4, 4, 4, 4, 4, 4], dim=64)
    m.load_state_dict(torch.load('/global/u2/k/kberard/SCGSR/Research/Diamond/stock_models/SCUNet/model_zoo/scunet_color_25.pth', map_location='cpu'))
    m.to(DEVICE)
    m.eval()
    models_dict['scunet_pre'] = m

    # Prepare results storage
    results_dict = {model: [] for model in models_to_run}
    results_ref = []

    print("\n=== Commencing Denoising Loop ===")
    for noisy_path in noisy_files:
        match = re.search(r"(\d+)\.h5$", noisy_path)
        if not match: continue
        sample_num = int(match.group(1))
        
        with h5py.File(noisy_path, 'r') as file:
            test_d = file['density'][:]
            
        # 1. Evaluate baseline Noisy dmc JSD
        jsd_ref = evaluate_and_enforce(test_d, ref_d_mean)
        results_ref.append((sample_num, jsd_ref))
        print(f"\n-> Sample {sample_num} loaded. Baseline JSD: {jsd_ref:.6e}")
        
        # 2. Iterate through Models
        for model_name in models_to_run:
            raw_denoised = run_model_inference(test_d, dft_d, model_name, models_dict)
            jsd_score = evaluate_and_enforce(raw_denoised, ref_d_mean)
            
            results_dict[model_name].append((sample_num, jsd_score))
            print(f"   ↳ {model_name} JSD = {jsd_score:.6e}")
            
            # Save Numpy Array
            output_file = os.path.join(base_dir, f"{os.path.splitext(os.path.basename(noisy_path))[0]}_{model_name}_denoised.npy")
            np.save(output_file, raw_denoised)

    generate_log_plots(results_dict, results_ref, DFT_vs_dmc, models_to_run)

if __name__ == "__main__":
    main()

/global/homes/k/kberard/.local/perlmutter/pytorch2.6.0/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading Base Data & DFT Reference...
Found 33 noisy files.
Pre-loading SCUNet into Memory on cuda...
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.0000